In [ ]:
!pip install -q pymupdf pandas transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.4 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF
import pandas as pd
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

# ==========================================
# 1. SETUP THE OPEN-SOURCE MODEL
# ==========================================
# We use Mistral-7B-Instruct-v0.3 as an example.
# Replace 'model_id' with the model you selected in the previous task if different.
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# Quantization config to run on Colab's free T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Create the inference pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,  # Short output needed (just JSON)
    temperature=0.1,     # Low temp for deterministic logic
    return_full_text=False
)

# ==========================================
# 2. DEFINE THE PROCESSING FUNCTIONS
# ==========================================

def get_page_metadata(text_content):
    """
    Sends page text to the LLM to determine if it's a new document and its type.
    """
    # System prompt to enforce JSON output
    sys_prompt = """
    You are a document classification assistant. Analyze the text provided.
    1. Determine if this page is the START of a new document (is_new_doc).
    2. Classify the document type (doc_type) from this list: [Resume, Contract, ID Proof, Invoice, Application, Other].

    Return the answer ONLY as a JSON object with keys "is_new_doc" (Yes/No) and "doc_type".
    Do not add any explanation or markdown.
    """

    user_prompt = f"Page Text:\n{text_content[:2000]}...\n\nJSON Output:" # Truncate to save tokens if needed

    # Format prompt for Mistral/Llama
    messages = [
        {"role": "user", "content": sys_prompt + "\n" + user_prompt}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    try:
        output = pipe(prompt)[0]['generated_text']
        # Extract JSON part (simple cleanup)
        output_clean = output.strip().replace("```json", "").replace("```", "")
        return json.loads(output_clean)
    except Exception as e:
        print(f"Error parsing model output: {e}")
        # Fallback default
        return {"is_new_doc": "No", "doc_type": "Unknown"}

# ==========================================
# 3. RUN THE PIPELINE
# ==========================================

# Replace with your actual file path
pdf_path = "/content/TB1.pdf"

doc = fitz.open(pdf_path)
final_metadata = []
page_in_doc_counter = 0

print("Starting processing...")

for page_num, page in enumerate(doc):
    text = page.get_text()

    # 1. Get analysis from Model
    analysis = get_page_metadata(text)

    is_new = analysis.get("is_new_doc", "No")
    doc_type = analysis.get("doc_type", "Unknown")

    # 2. Logic for page_in_doc counter
    if is_new == "Yes":
        page_in_doc_counter = 0
    else:
        # If it's the very first page of the PDF, force it to 0
        if page_num == 0:
            page_in_doc_counter = 0
        else:
            page_in_doc_counter += 1

    # 3. Store Data
    entry = {
        "page": page_num,
        "is_new_doc": is_new,
        "doc_type": doc_type,
        "page_in_doc": page_in_doc_counter
    }
    final_metadata.append(entry)

    print(f"Processed Page {page_num}: {is_new} - {doc_type}")

# ==========================================
# 4. EXPORT RESULTS
# ==========================================

df = pd.DataFrame(final_metadata)
print("\nFinal Pipeline Output:")
print(df.head(10))

# Save to JSON as requested
df.to_json("page_metadata.json", orient="records", indent=4)

Loading mistralai/Mistral-7B-Instruct-v0.3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

... (18 outputs left out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed Page 4: Yes - Contract


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed Page 5: Yes - Contract
Processed Page 6: Yes - Contract

Final Pipeline Output:
   page is_new_doc                doc_type  page_in_doc
0     0        Yes  Mortgage Loan Estimate            0
1     1         No                 Payslip            1
2     2        Yes                Contract            0
3     3        Yes                Contract            0
4     4        Yes                Contract            0
5     5        Yes                Contract            0
6     6        Yes                Contract            0
